## 中断

#### 静态中断

In [ ]:
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    output: str

def a(state: State):
    print("节点 a 被调用")
    return {
        "output": "node a output",
    }

def b(state: State):
    print("节点 b 被调用")
    return {
        "output": "node b output",
    }

def final(state: State):
    return {
        "output": "final output",
    }


builder = StateGraph(State)
builder.add_node(a)
builder.add_node(b)
builder.add_node(final)
builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("b", "final")
builder.add_edge("final", END)
graph = builder.compile(
    checkpointer=InMemorySaver(),
    interrupt_before=["a", "b"],
    interrupt_after=["a", "b"],
)

config = {"configurable": {"thread_id": "5-3-1"}}
print("======第一次调用======")
result = graph.invoke({}, config=config)
print(result)

print("======第二次调用======")
result = graph.invoke(None, config=config)
print(result)

print("======第三次调用======")
result = graph.invoke(None, config=config)
print(result)

print("======第四次调用======")
result = graph.invoke(None, config=config)
print(result)

#### 并发静态中断

以 superstep 作为中断依据，相同会被同时中断

In [ ]:
from operator import add
from typing import TypedDict, Annotated, List

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    output: Annotated[List[str], add]

def a(state: State):
    print("节点 a 被调用")
    return {
        "output": ["a"],
    }

def b(state: State):
    print("节点 b 被调用")
    return {
        "output": ["b"],
    }

def c(state: State):
    print("节点 c 被调用")
    return {
        "output": ["c"],
    }

def d(state: State):
    print("节点 d 被调用")
    return {
        "output": ["d"],
    }

def final(state: State):
    return {
        "output": ["final"],
    }


builder = StateGraph(State)
builder.add_node(a)
builder.add_node(b)
builder.add_node(c)
builder.add_node(d)
builder.add_node(final)
builder.add_edge(START, "a")
builder.add_edge(START, "c")
builder.add_edge("a", "b")
builder.add_edge("c", "d")
builder.add_edge("b", "final")
builder.add_edge("d", "final")
builder.add_edge("final", END)
graph = builder.compile(
    checkpointer=InMemorySaver(),
    interrupt_before=["a", "b", "final"],
)

config = {"configurable": {"thread_id": "5-3-2"}}
print("======第一次调用======")
result = graph.invoke({}, config=config)
print(result)

print("======第二次调用======")
result = graph.invoke(None, config=config)
print(result)

print("======第三次调用======")
result = graph.invoke(None, config=config)
print(result)

print("======第四次调用======")
result = graph.invoke(None, config=config)
print(result)